## Canny Edge 기반 실시간 숫자 인식 및 버저 제어

### 1. 실습 목적
Raspbot2 카메라 영상에서 Canny Edge Detection으로 사각형을 검출하고, LeNet ONNX 모델로 내부 숫자를 인식하여 버저를 제어한다.

### 2. 처리 과정
1. **Grayscale:** 카메라 영상을 흑백으로 변환
2. **GaussianBlur:** 영상 노이즈 제거
3. **Canny Edge:** 경계선 검출
4. **findContours / approxPolyDP:** 윤곽선 및 사각형 검출
5. **ROI Extraction:** 사각형 내부 추출 및 테두리 제외
6. **Preprocessing:** 32×32 크기 변환, 색 반전, 정규화
7. **LeNet ONNX:** 숫자 0~9 분류
8. **Buzzer:** 동일 숫자 3프레임 연속 인식 시 해당 숫자만큼 작동

### 3. 전체 구조
**Camera → GaussianBlur → Canny → Contours → ROI → LeNet ONNX → Buzzer**

### 4. 한계점
- 숫자가 아닌 사각형도 숫자로 오인식할 가능성 존재
- 조명 및 카메라 흔들림에 따라 사각형 검출 성능 저하 가능

In [ ]:

import cv2, time, numpy as np, onnxruntime as ort
from picamera2 import Picamera2
from Raspbot_Lib import Raspbot
from IPython.display import display, Image

model = ort.InferenceSession("my_lenet.onnx")
input_name = model.get_inputs()[0].name
bot = Raspbot()
cam = Picamera2()
cam.configure(cam.create_preview_configuration(main={"size": (640, 480), "format": "RGB888"}))
cam.start()

screen = None
prev, count, beeped, missing = None, 0, False, 0

try:
    while True:
        frame = cv2.cvtColor(cam.capture_array(), cv2.COLOR_RGB2BGR)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 50, 150)
        contours, _ = cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        found = False

        for c in sorted(contours, key=cv2.contourArea, reverse=True):
            if cv2.contourArea(c) < 3000: continue
            if len(cv2.approxPolyDP(c, 0.03 * cv2.arcLength(c, True), True)) != 4: continue
            x, y, w, h = cv2.boundingRect(c)
            if not 0.6 < w/h < 1.6: continue

            m = int(min(w, h) * 0.12)
            roi = gray[y+m:y+h-m, x+m:x+w-m]
            if roi.size == 0: continue

            digit = cv2.resize(255-roi, (32, 32)).astype(np.float32) / 255
            digit = digit.reshape(1, 1, 32, 32)
            num = int(np.argmax(model.run(None, {input_name: digit})[0]))
            found = True

            count = count + 1 if num == prev else 1
            prev, missing = num, 0
            cv2.rectangle(frame, (x,y), (x+w,y+h), (0,255,0), 2)
            cv2.putText(frame, f"Detected: {num}", (x,max(30,y-15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)
            break

        if not found:
            missing += 1
            count, prev = 0, None
            cv2.putText(frame, "Searching...", (20,40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,255), 2)
            if missing >= 6: beeped = False

        _, buf = cv2.imencode(".jpg", frame)
        img = Image(data=buf.tobytes())
        if screen is None: screen = display(img, display_id=True)
        else: screen.update(img)

        if found and count >= 3 and not beeped:
            beeped = True
            for _ in range(num):
                bot.Ctrl_BEEP_Switch(1)
                time.sleep(0.1)
                bot.Ctrl_BEEP_Switch(0)
                time.sleep(0.15)

        time.sleep(0.05)

except KeyboardInterrupt:
    pass
finally:
    bot.Ctrl_BEEP_Switch(0)
    cam.stop()
    cam.close()



## OpenCV 및 LeNet ONNX 기반 실시간 숫자 인식 및 버저 제어

### 1. 실습 목적
Raspbot2의 카메라를 이용하여 사각형 내부의 손글씨 숫자를 검출하고, 학습된 LeNet 모델을 통해 숫자(0~9)를 분류한다. 인식된 숫자를 카메라 화면에 표시하고 해당 숫자만큼 버저를 작동시킨다.

### 2. 사용 기술
- **OpenCV:** 영상 전처리 및 사각형 영역 검출
- **Picamera2:** 카메라 영상 입력 (640×480)
- **LeNet (ONNX Runtime):** 숫자 이미지 분류
- **Raspbot_Lib:** 버저 제어

### 3. 영상처리 및 숫자 인식 과정

1. **Grayscale:** 입력 영상을 흑백으로 변환
2. **GaussianBlur:** 영상 노이즈 제거
3. **Adaptive Threshold:** 조명 변화에 대응하는 이진화
4. **Morphological Closing:** 끊어진 테두리 연결
5. **Contour Detection:** 윤곽선 및 사각형 검출
6. **ROI Extraction:** 사각형 내부 영역 추출 및 테두리 제외
7. **Image Preprocessing:** 32×32 크기 변환, 색 반전 및 정규화
8. **LeNet ONNX Inference:** 숫자 0~9 분류

### 4. 인식 결과 및 버저 제어
- 숫자 검출 시 화면에 `Detected: N` 표시
- 같은 숫자가 **3프레임 연속 인식되면** 숫자 N만큼 버저 작동
- 숫자가 검출되지 않으면 `Searching...` 표시
- 검출 실패가 6프레임 이상 지속되면 버저 재작동 가능 상태로 초기화

### 5. 전체 처리 흐름

**Camera → OpenCV Preprocessing → Rectangle Detection → ROI Extraction → 32×32 Normalization → LeNet ONNX → Number Prediction → Buzzer Control**

### 6. 구현 시 고려사항
- 숫자가 아닌 사각형도 숫자로 잘못 분류될 수 있음
- 조명, 촬영 거리 및 카메라 흔들림에 따라 검출 성능이 달라질 수 있음
- 3프레임 연속 인식 조건을 통해 버저의 일시적인 오작동을 줄임

In [ ]:

import cv2, time, numpy as np, onnxruntime as ort
from picamera2 import Picamera2
from Raspbot_Lib import Raspbot
from IPython.display import display, Image

model = ort.InferenceSession("my_lenet.onnx")
input_name = model.get_inputs()[0].name

bot = Raspbot()
bot.Ctrl_BEEP_Switch(0)

cam = Picamera2()
cam.configure(cam.create_preview_configuration(
    main={"size": (640, 480), "format": "RGB888"}
))
cam.start()

screen = None
prev, count, beeped, missing = None, 0, False, 0

try:
    while True:
        frame = cv2.cvtColor(cam.capture_array(), cv2.COLOR_RGB2BGR)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # 1. 블러 + 적응형 이진화
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        binary = cv2.adaptiveThreshold(
            blur, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV, 31, 10
        )

        # 2. 끊어진 검은 테두리 연결
        kernel = np.ones((3, 3), np.uint8)
        binary = cv2.morphologyEx(
            binary, cv2.MORPH_CLOSE, kernel
        )

        # 3. 네모 박스 찾기
        contours, _ = cv2.findContours(
            binary, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE
        )

        found = False

        for c in sorted(contours, key=cv2.contourArea, reverse=True):
            area = cv2.contourArea(c)
            if area < 3000 or area > gray.size * 0.85:
                continue

            approx = cv2.approxPolyDP(
                c, 0.03 * cv2.arcLength(c, True), True
            )

            if len(approx) != 4:
                continue

            x, y, w, h = cv2.boundingRect(approx)
            if not 0.6 < w / h < 1.6:
                continue

            # 4. 검은 테두리 제외하고 숫자 추출
            m = int(min(w, h) * 0.12)
            roi = gray[y+m:y+h-m, x+m:x+w-m]

            if roi.size == 0:
                continue

            # 5. LeNet 입력 전처리
            digit = cv2.resize(255-roi, (32, 32))
            digit = digit.astype(np.float32) / 255.0
            digit = digit.reshape(1, 1, 32, 32)

            # 6. ONNX 숫자 인식
            logits = model.run(None, {input_name: digit})[0]
            num = int(np.argmax(logits))
            found = True

            count = count + 1 if num == prev else 1
            prev, missing = num, 0

            # 7. 인식 결과 표시
            cv2.rectangle(
                frame, (x, y), (x+w, y+h),
                (0, 255, 0), 2
            )

            cv2.putText(
                frame, f"Detected: {num}",
                (x, max(30, y-15)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9, (0, 255, 0), 2
            )
            break

        if not found:
            missing += 1
            count, prev = 0, None

            cv2.putText(
                frame, "Searching...", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1, (0, 255, 255), 2
            )

            if missing >= 6:
                beeped = False

        # 8. Jupyter 카메라 화면 갱신
        _, buf = cv2.imencode(".jpg", frame)
        img = Image(data=buf.tobytes())

        if screen is None:
            screen = display(img, display_id=True)
        else:
            screen.update(img)

        # 9. 숫자 3프레임 연속 인식 시 버저
        if found and count >= 3 and not beeped:
            beeped = True

            for _ in range(num):
                bot.Ctrl_BEEP_Switch(1)
                time.sleep(0.10)
                bot.Ctrl_BEEP_Switch(0)
                time.sleep(0.15)

        time.sleep(0.05)

except KeyboardInterrupt:
    print("프로그램 종료")

finally:
    bot.Ctrl_BEEP_Switch(0)
    cam.stop()
    cam.close()
    print("카메라 종료 완료")


# OpenCV 및 딥러닝 기반 실시간 숫자 인식 시스템

## 1. 프로젝트 개요

Raspbot2 카메라로 사각형 내부의 숫자를 검출하고, LeNet ONNX 모델을 통해 숫자(0~9)를 분류한 후 해당 숫자만큼 버저를 작동시키는 시스템을 구현한다.

**전체 구조**

`Camera → OpenCV → ROI 추출 → 32×32 전처리 → LeNet ONNX → Buzzer`

## 2. 전통적인 OpenCV 영상처리 기법

### ① Canny Edge Detection
- 이미지의 밝기 변화가 큰 부분에서 경계선을 검출하는 알고리즘
- `GaussianBlur`로 노이즈를 줄인 뒤 `Canny` 적용
- `findContours`로 윤곽선을 추출하고 `approxPolyDP`로 사각형 여부 판별
- **한계:** 조명 변화, 영상 흔들림, 끊어진 테두리에 취약할 수 있음

### ② Adaptive Thresholding (적응형 이진화)
- 주변 픽셀의 밝기를 기준으로 영상을 흑백으로 변환
- 이미지 전체에 동일한 임계값을 적용하는 방식보다 불균일한 조명에 대응하기 유리
- 검은색 숫자와 사각형 테두리를 분리하는 데 활용

### ③ Morphological Operations (형태학적 연산)
- **Dilation(팽창):** 흰색 영역 확장
- **Erosion(침식):** 흰색 영역 축소
- **Closing:** 팽창 → 침식, 끊어진 테두리 연결
- **Opening:** 침식 → 팽창, 작은 잡음 제거

**개선 구조**

`Grayscale → GaussianBlur → Adaptive Threshold → Morphological Closing → Contour Detection`

## 3. 심화 영상처리 기술

### ① Perspective Transform (원근 변환)
- 사각형의 네 꼭짓점을 이용하여 기울어진 이미지를 정면 영상으로 변환
- `getPerspectiveTransform()`과 `warpPerspective()` 사용
- 카메라 촬영 각도에 따른 숫자 왜곡 감소에 활용

### ② ROI 정규화 및 숫자 전처리
- 검출된 사각형 내부에서 검은 테두리 제거
- 밝기 보정 및 색 반전
- 숫자의 가로세로 비율을 유지하며 32×32 이미지 생성
- 학습 데이터와 유사한 입력 형태로 변환

### ③ 후보 점수화 및 객체 추적
- 여러 사각형 후보의 형태, 크기, 배경 밝기 등을 평가
- 가장 적합한 숫자 영역 선택
- 이전 프레임의 위치와 예측 결과를 활용하여 일시적인 오검출 감소

## 4. 딥러닝 기반 심화 기술

### ① LeNet CNN
- MNIST 손글씨 숫자 데이터 기반 숫자 분류 모델
- 입력: 32×32 흑백 이미지
- 출력: 숫자 0~9
- ONNX Runtime을 통해 Raspbot2에서 추론 수행

### ② YOLO Object Detection
- 딥러닝을 이용해 영상에서 객체의 위치를 검출하는 기술
- 숫자가 적힌 종이의 위치를 학습하여 검출 가능
- 기존 Canny 및 Adaptive Threshold 방식의 사각형 검출을 대체할 수 있음
- 별도의 객체 탐지 데이터셋과 학습 과정 필요

### ③ YOLO + LeNet 통합 구조

`Camera → YOLO(숫자 영역 검출) → OpenCV(전처리) → LeNet(숫자 분류) → Buzzer`

- **YOLO:** 숫자가 적힌 영역의 위치 검출
- **OpenCV:** 영역 추출 및 이미지 정규화
- **LeNet:** 숫자 0~9 분류
- **Raspbot2:** 인식된 숫자에 따른 버저 제어

## 5. 기술별 비교

| 기술 | 주요 역할 | 특징 |
|---|---|---|
| Canny | 경계선 검출 | 간단하고 계산량이 적음 |
| Adaptive Threshold | 밝기 기반 영역 분리 | 조명 변화 대응 |
| Morphology | 잡음 제거 및 선 연결 | 검출 영역 보정 |
| Perspective Transform | 기울기·원근 왜곡 보정 | 입력 이미지 정렬 |
| Contour Scoring | 후보 영역 평가 | 잘못된 사각형 선택 감소 |
| LeNet CNN | 숫자 분류 | 경량 신경망 |
| YOLO | 객체 위치 검출 | 학습 기반 객체 탐지 |

## 6. 실전 적용 및 향후 개선 방향

**현재 구현**
- OpenCV를 이용한 사각형 검출
- LeNet ONNX 기반 숫자 인식
- 숫자 결과 실시간 표시
- 동일 숫자 3프레임 연속 인식 시 버저 제어

**향후 개선**
1. Canny와 Adaptive Threshold의 검출 성능 비교
2. 원근 변환을 이용한 기울어진 숫자 보정
3. 후보 영역 점수화 및 숫자 분류 신뢰도 분석
4. 실제 환경에서 발생하는 오인식 데이터 수집
5. YOLO 기반 숫자 영역 탐지와 LeNet 분류 모델 통합

## 7. 결론

본 시스템은 **전통적인 OpenCV 영상처리와 CNN 기반 딥러닝을 결합한 하이브리드 컴퓨터 비전 시스템**이다.

현재는 OpenCV 기반 사각형 검출과 LeNet 숫자 분류를 구현하였으며, 향후 원근 보정, 후보 추적 및 YOLO 객체 탐지를 적용하여 실제 로봇 주행 환경에서의 인식 안정성을 개선할 수 있다.